# CytoVI Annotation & Baseline Comparison

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
from scvi.external import CYTOVI
from scvi.external.cytovi import scale as cytovi_scale, transform_arcsinh
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

print(f'scvi-tools: {scvi.__version__}')

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
SOURCE_CACHE     = CACHE_DIR / 'adata_all_annotated.h5ad'
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated.h5ad'
MODEL_DIR        = CACHE_DIR / 'cytovi_model'
ASINH_SCALE      = 5.0
LEIDEN_RES       = 0.5
FDR_THRESH       = 0.05
spatial_rep      = 'spatial_asinh5_top500var'

## 1. Load data

In [ ]:
# Load annotated data (with spatial features + cell_type_annot from annotation_comparison)
adata = sc.read_h5ad(SOURCE_CACHE)
print(adata)
print('\nLayers  :', list(adata.layers.keys()))
print('\nSamples and cell systems:')
print(adata.obs.groupby(['sample', 'cell_system'], observed=True).size().to_string())

In [ ]:
# Cell count table: samples × cell_system / time / condition
tbl = (
    adata.obs
    .groupby(['sample', 'cell_system', 'time', 'condition'], observed=True)
    .size()
    .reset_index(name='n_cells')
    .sort_values(['sample','cell_system', 'time', 'condition', ])
    .reset_index(drop=True)
)
display(tbl.style.set_caption('Cell counts per sample by cell system, time, and treatment'))

## 2. CytoVI model

In [ ]:
# Per-batch arcsinh transform + min-max scaling (following CytoVI tutorial)
from scvi.external.cytovi import merge_batches

batches = adata.obs['cell_system'].unique().tolist()
adata_batches = []

for batch in batches:
    ad = adata[adata.obs['cell_system'] == batch].copy()

    # Reconstruct raw counts from log1p layer and store as 'raw'
    ad.layers['raw'] = np.expm1(np.array(ad.layers['log1p'], dtype=np.float32))

    # Arcsinh transform per batch (cofactor=5 for MPX/CyTOF-like data)
    transform_arcsinh(ad, global_scaling_factor=ASINH_SCALE)
    # Min-max scale per batch
    cytovi_scale(ad)

    print(f'{batch}: {ad.n_obs} cells, layers = {list(ad.layers.keys())}')
    adata_batches.append(ad)

print(f'\nTotal batches: {len(adata_batches)}')

In [ ]:
# Merge per-batch scaled data back together
# Preserve original obs columns and obsm through merge
obs_cols_to_keep = [c for c in adata.obs.columns if c not in ['batch']]
obsm_keys = list(adata.obsm.keys())

adata_merged = merge_batches(adata_batches)
print(f'Merged: {adata_merged.shape}')
print(f'Layers: {list(adata_merged.layers.keys())}')

# Restore obs metadata from original adata (merge_batches only keeps 'batch')
shared_idx = adata_merged.obs_names.intersection(adata.obs_names)
for col in obs_cols_to_keep:
    if col in adata.obs.columns and col not in adata_merged.obs.columns:
        adata_merged.obs[col] = adata.obs.loc[shared_idx, col]

# Restore obsm (spatial features, etc.)
for key in obsm_keys:
    if key not in adata_merged.obsm:
        adata_merged.obsm[key] = adata.obsm[key][adata.obs_names.isin(shared_idx)]

adata = adata_merged
print(f'\nFinal adata: {adata.shape}')
print(f'obs columns: {list(adata.obs.columns)}')

In [ ]:
# Uncorrected UMAP (before CytoVI) to visualize batch effects
adata.X = adata.layers['scaled']
sc.pp.neighbors(adata, use_rep='X', n_neighbors=20)
sc.tl.umap(adata)
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()



In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
sc.pl.umap(adata, color='cell_system', ax=axes[0], show=False, title='Uncorrected — Batch (cell system)')
sc.pl.umap(adata, color='sample',    ax=axes[1], show=False, title='Uncorrected — Sample')
sc.pl.umap(adata, color='condition', ax=axes[2], show=False, title='Uncorrected — Condition')
sc.pl.umap(adata, color='time', ax=axes[3], show=False, title='Uncorrected — Time')

plt.tight_layout()
plt.show()

In [ ]:
# Setup and train CytoVI (or load from cache)
import torch
use_gpu = torch.cuda.is_available()
print(f'GPU available: {use_gpu}')
if use_gpu:
    print(f'GPU device: {torch.cuda.get_device_name(0)}')

# Use 'batch' key created by merge_batches (maps to cell_system)
if MODEL_DIR.exists():
    print(f'Loading cached model from {MODEL_DIR}')
    CYTOVI.setup_anndata(adata, layer='scaled', batch_key='batch')
    model = CYTOVI.load(MODEL_DIR, adata=adata)
else:
    CYTOVI.setup_anndata(adata, layer='scaled', batch_key='batch')
    model = CYTOVI(adata, protein_likelihood='normal')
    model.train(
        batch_size=1024,
        n_epochs_kl_warmup=50,
        accelerator='gpu' if use_gpu else 'cpu',
    )
    model.save(MODEL_DIR, overwrite=True)
    print(f'Model saved -> {MODEL_DIR}')

print(model)

In [ ]:
# Plot training history
if hasattr(model, 'history') and 'elbo_train' in model.history:
    plt.figure(figsize=(6, 4))
    plt.plot(model.history['elbo_train'], label='Train')
    plt.plot(model.history['elbo_validation'], label='Validation')
    plt.xlabel('Epochs')
    plt.ylabel('ELBO')
    plt.legend()
    plt.title('Training vs Validation ELBO')
    plt.tight_layout()
    plt.show()
else:
    print('Training history not available (model loaded from cache)')

In [ ]:
# Extract latent representation and denoised expression
adata.obsm['X_CytoVI'] = model.get_latent_representation()
adata.layers['imputed'] = model.get_normalized_expression()
print(f'Latent shape: {adata.obsm["X_CytoVI"].shape}')
print(f'Imputed layer shape: {adata.layers["imputed"].shape}')

# Neighbors + UMAP on CytoVI latent
sc.pp.neighbors(adata, use_rep='X_CytoVI', n_neighbors=20)
sc.tl.umap(adata, min_dist=0.3)
adata.obsm['X_umap_cytovi'] = adata.obsm['X_umap'].copy()

In [ ]:
# CytoVI-corrected UMAP overview
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
sc.pl.umap(adata, color='cell_system',     ax=axes[0], show=False, title='CytoVI — Batch (cell system)')
sc.pl.umap(adata, color='sample',    ax=axes[1], show=False, title='CytoVI — Sample')
sc.pl.umap(adata, color='condition', ax=axes[2], show=False, title='CytoVI — Condition')
sc.pl.umap(adata, color='time',      ax=axes[3], show=False, title='CytoVI — Time')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize denoised (imputed) expression of key markers on UMAP
key_markers = ['CD3e', 'CD4', 'CD8', 'CD19', 'CD20']
key_markers = [m for m in key_markers if m in adata.var_names]
sc.pl.umap(adata, color=key_markers, layer='imputed', ncols=5, cmap='mako')

In [ ]:
## 3. Cluster & Annotate

In [ ]:
ANNOTATED_CACHE = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'
adata = sc.read_h5ad(ANNOTATED_CACHE)

In [ ]:
# Leiden clustering on CytoVI latent
sc.pp.neighbors(adata, use_rep='X_CytoVI', n_neighbors=20)
sc.tl.umap(adata, min_dist=0.3)

sc.tl.leiden(adata, resolution=0.5, key_added='leiden')
print(f'Clusters: {adata.obs["leiden"].nunique()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title='Leiden clusters')
sc.pl.umap(adata, color='cell_system',  ax=axes[1], show=False, title='Cell System')
plt.tight_layout()
plt.show()

In [ ]:
# Dotplot — denoised expression per cluster for annotation
marker_groups = {
    'T cell': ['CD3e', 'TCRab', 'CD5', 'CD7', 'CD2'],
    'CD4 T':  ['CD4',],
    'CD8 T':  ['CD8', 'CD57', 'KLRG1'],
    'B cell': ['CD19', 'CD20', 'CD22', 'IgM', 'IgD'],
}
markers_f = {k: [m for m in v if m in adata.var_names] for k, v in marker_groups.items()}
markers_f = {k: v for k, v in markers_f.items() if v}

sc.pl.dotplot(
    adata, var_names=markers_f, groupby='leiden',
    layer='imputed', standard_scale='var',
    title='CytoVI — denoised marker expression per cluster',
    figsize=(16, 6),
)

In [ ]:
# --- ANNOTATION DICT: map leiden cluster -> cell type ---
cluster_annotation = {
    '0': 'CD4',   '1': 'CD4',   '2': 'CD8',   '3': 'B',
    '4': 'B',   '5': 'CD4',   '6': 'CD4',   '7': 'B',
    '8': 'Doublets',   '9': 'CD4/CD8',  '10': 'CD8',  '11': 'B',
   '12': 'B',  '13': 'CD4',  '14': 'B',  '15': 'B',
   '16': 'B',  '17': 'B'
}

In [ ]:
# Apply annotation
assert all(v for v in cluster_annotation.values()), 'Fill in all cluster labels first'
adata.obs['cell_type_annot'] = adata.obs['leiden'].map(cluster_annotation)
unmapped = adata.obs['cell_type_annot'].isna().sum()
if unmapped:
    print(f'WARNING: {unmapped} cells have unmapped clusters')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sc.pl.umap(adata, color='cell_type_annot', ax=axes[0], show=False, title='Cell type annotation')
sc.pl.umap(adata, color='batch',           ax=axes[1], show=False, title='Batch (cell system)')
sc.pl.umap(adata, color='condition',       ax=axes[2], show=False, title='Condition')
plt.tight_layout()
plt.show()

print(adata.obs['cell_type_annot'].value_counts())

In [ ]:
# Save annotated cache
adata.write_h5ad(ANNOTATED_CACHE)
print(f'Saved -> {ANNOTATED_CACHE}')

## 4. Baseline comparison: T cells, 6h Mock

Compare T cells (CD4 and CD8 separately) in **NALM-6 + healthy T** vs **healthy B + healthy T** at 6h Mock.

In [ ]:
# Load annotated adata from cache (run this cell to start directly from section 4)
if ANNOTATED_CACHE.exists():
    adata = sc.read_h5ad(ANNOTATED_CACHE)
    print(f'Loaded from cache: {ANNOTATED_CACHE}')
else:
    raise FileNotFoundError(f'Run sections 1-3 first to create {ANNOTATED_CACHE}')
print(adata.obs['cell_type_annot'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
sc.pl.umap(adata, color='cell_system',     ax=axes[0], show=False, title='CytoVI — Batch (cell system)')
sc.pl.umap(adata, color='sample',    ax=axes[1], show=False, title='CytoVI — Sample')
sc.pl.umap(adata, color='condition', ax=axes[2], show=False, title='CytoVI — Condition')
sc.pl.umap(adata, color='time',      ax=axes[3], show=False, title='CytoVI — Time')
plt.tight_layout()
plt.show()

## 5. Cell composition & B/T ratio over time

All cell systems, all timepoints. Doublets and CD4/CD8 double-positives removed.

In [ ]:
# --- Remove doublets & define filtered AnnData for compositional analyses ---
DOUBLET_LABELS = {'Doublets', 'CD4/CD8'}
adata_filtered = adata[
    ~adata.obs['cell_type_annot'].isin(DOUBLET_LABELS)
].copy()
print(f'Retained {adata_filtered.n_obs} / {adata.n_obs} cells after doublet removal')
print(adata_filtered.obs['cell_type_annot'].value_counts())

COND_PALETTE = {'Mock': '#5975A4', 'Blinatumomab': '#E8655A'}

sample_info = (
    adata_filtered.obs
    .groupby('sample', observed=True)[['condition', 'time', 'cell_system']]
    .first()
)

In [ ]:
# --- 7a. Cell type composition over time per cell system ---
# B, CD4, CD8 fractions; lines connect mean across samples per condition
ct_counts = (
    adata_filtered.obs
    .groupby(['sample', 'cell_type_annot'], observed=True)
    .size()
    .unstack(fill_value=0)
)
ct_counts.columns = ct_counts.columns.astype(str)  # drop CategoricalIndex
ct_frac = ct_counts.div(ct_counts.sum(axis=1), axis=0)
ct_frac = ct_frac.join(sample_info)
ct_frac['time_h'] = ct_frac['time'].astype(str).str.replace('h', '').astype(int)

ct_cols = [c for c in ct_frac.columns if c not in ['condition', 'time', 'cell_system', 'time_h']]
ct_melt = ct_frac.reset_index().melt(
    id_vars=['sample', 'condition', 'time', 'time_h', 'cell_system'],
    value_vars=ct_cols, var_name='cell_type', value_name='fraction',
)

systems = sorted(ct_melt['cell_system'].unique())
cell_types = sorted(ct_melt['cell_type'].unique())  # B, CD4, CD8
n_sys = len(systems)

fig, axes = plt.subplots(1, n_sys, figsize=(5 * n_sys, 5), sharey=True)
if n_sys == 1:
    axes = [axes]

CT_PALETTE = {'B': '#4C72B0', 'CD4': '#DD8452', 'CD8': '#55A868'}

for ax, sys in zip(axes, systems):
    sub = ct_melt[ct_melt['cell_system'] == sys]
    for ct in cell_types:
        color = CT_PALETTE.get(ct, 'grey')
        for cond, ls in zip(['Mock', 'Blinatumomab'], ['-', '--']):
            d = sub[(sub['cell_type'] == ct) & (sub['condition'] == cond)]
            if d.empty:
                continue
            means = d.groupby('time_h')['fraction'].mean()
            ax.plot(means.index, means.values, marker='o', ls=ls, color=color,
                    label=f'{ct} ({cond})', markersize=6, lw=2)
            ax.scatter(d['time_h'], d['fraction'], color=color, alpha=0.3, s=20, zorder=1)

    ax.set_title(sys, fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (h)')
    ax.set_xticks([6, 48])
    ax.set_ylabel('Fraction of cells' if ax == axes[0] else '')
    sns.despine(ax=ax)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, title='Cell type (condition)', frameon=True,
           bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=9)
fig.suptitle('Cell type composition over time — solid=Mock, dashed=Blinatumomab',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# --- 7b. B/T ratio over time per cell system ---
# T = all T subtypes (CD4 + CD8); B vs T comparison
B_LINEAGE = {'B'}
T_LINEAGE = {'CD4', 'CD8'}

obs = adata_filtered.obs
bt_df = (
    obs.groupby('sample', observed=True)
    .apply(
        lambda g: pd.Series({
            'n_B': g['cell_type_annot'].isin(B_LINEAGE).sum(),
            'n_T': g['cell_type_annot'].isin(T_LINEAGE).sum(),
            'total': len(g),
        }),
        include_groups=False,
    )
    .join(sample_info)
)
bt_df['BT_ratio']  = bt_df['n_B'] / bt_df['n_T'].replace(0, np.nan)
bt_df['log2_BT']   = np.log2(bt_df['BT_ratio'].replace(0, np.nan))
bt_df['frac_B']    = bt_df['n_B'] / bt_df['total']
bt_df['frac_T']    = bt_df['n_T'] / bt_df['total']
bt_df['time_h']    = bt_df['time'].astype(str).str.replace('h', '').astype(int)

systems = sorted(bt_df['cell_system'].unique())
n_sys   = len(systems)

# Row 1 — log2(B/T) over time
fig, axes = plt.subplots(1, n_sys, figsize=(5 * n_sys, 5), sharey=True)
if n_sys == 1:
    axes = [axes]

for ax, sys in zip(axes, systems):
    sub = bt_df[bt_df['cell_system'] == sys].reset_index()
    for cond, color, marker in zip(
        ['Mock', 'Blinatumomab'],
        [COND_PALETTE['Mock'], COND_PALETTE['Blinatumomab']],
        ['o', 's'],
    ):
        d = sub[sub['condition'] == cond]
        if d.empty:
            continue
        ax.scatter(d['time_h'], d['log2_BT'], color=color, marker=marker,
                   s=60, zorder=3, edgecolors='white', linewidths=0.5)
        means = d.groupby('time_h')['log2_BT'].mean()
        ax.plot(means.index, means.values, color=color, marker=marker,
                ls='-', lw=2, markersize=10, label=cond, zorder=2)
    ax.axhline(y=0, color='grey', ls='--', lw=1, alpha=0.5)
    ax.set_title(sys, fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (h)')
    ax.set_xticks([6, 48])
    ax.set_ylabel('log\u2082(B / T)' if ax == axes[0] else '')
    if ax == axes[-1]:
        ax.legend(title='Condition', frameon=True)
    sns.despine(ax=ax)

fig.suptitle('B/T ratio over time per cell system', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Row 2 — B and T fractions separately
fig, axes = plt.subplots(1, n_sys, figsize=(5 * n_sys, 5), sharey=True)
if n_sys == 1:
    axes = [axes]

for ax, sys in zip(axes, systems):
    sub = bt_df[bt_df['cell_system'] == sys].reset_index()
    for lineage, color, frac_col in [
        ('B (CD19/CD20)', '#4C72B0', 'frac_B'),
        ('T (CD4+CD8)',   '#DD8452', 'frac_T'),
    ]:
        for cond, ls in [('Mock', '-'), ('Blinatumomab', '--')]:
            d = sub[sub['condition'] == cond]
            if d.empty:
                continue
            means = d.groupby('time_h')[frac_col].mean()
            ax.plot(means.index, means.values, color=color, ls=ls, marker='o',
                    lw=2, markersize=8, label=f'{lineage} ({cond})')
            ax.scatter(d['time_h'], d[frac_col], color=color, alpha=0.3, s=25, zorder=1)
    ax.set_title(sys, fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (h)')
    ax.set_xticks([6, 48])
    ax.set_ylabel('Fraction of cells' if ax == axes[0] else '')
    sns.despine(ax=ax)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, title='', frameon=True,
           bbox_to_anchor=(1.02, 0.5), loc='center left', fontsize=9)
fig.suptitle('B & T fractions over time — solid=Mock, dashed=Blinatumomab',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Subset to 6h Mock, T cells only (CD4 + CD8), healthy-T systems only
mask_6h_mock_t = (
    (adata.obs['time'] == '6h') &
(adata.obs['condition'] == 'Mock') &
    (adata.obs['cell_type_annot'].isin(['CD4', 'CD8','B'])) &
    (adata.obs['cell_system'].isin(['NALM-6 + healthy T', 'healthy B + healthy T']))
)
adata_6h = adata[mask_6h_mock_t].copy()

print(f'6h Mock T cells (healthy-T systems): {adata_6h.n_obs}')
pd.crosstab(index=adata_6h.obs['sample'], columns=[adata_6h.obs['cell_system'], adata_6h.obs['cell_type_annot']])

In [ ]:
sc.pl.umap(adata_6h, color=['CD3e','CD19','CD20','cell_system','condition'],show=False)


In [ ]:
# Helper functions

def diff_analysis(X_df, group_mask):
    """Mann-Whitney U test between two groups (mask=True -> group A)."""
    stats = []
    for feat in X_df.columns:
        x_a = X_df.loc[group_mask, feat].values
        x_b = X_df.loc[~group_mask, feat].values
        md  = x_a.mean() - x_b.mean()
        _, pval = mannwhitneyu(x_a, x_b, alternative='two-sided')
        stats.append({'feature': feat, 'mean_diff': md, 'pval': pval})
    df = pd.DataFrame(stats).set_index('feature')
    df['pval_adj'] = multipletests(df['pval'], method='fdr_bh')[1]
    df['neg_log10_padj'] = -np.log10(df['pval_adj'].replace(0, np.finfo(float).tiny))
    return df.sort_values('mean_diff', ascending=False)


def volcano_plot(da, diff_thresh, y_cap=300, fdr_thresh=FDR_THRESH,
                 label_a='Group A', label_b='Group B', title='Volcano', n_label=12):
    sig  = da['pval_adj'] < fdr_thresh
    up   = sig & (da['mean_diff'] >  diff_thresh)
    down = sig & (da['mean_diff'] < -diff_thresh)
    ns   = ~(up | down)
    y    = da['neg_log10_padj'].clip(upper=y_cap)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(da.loc[ns,   'mean_diff'], y[ns],   c='lightgrey', s=15, alpha=0.5, label='NS')
    ax.scatter(da.loc[up,   'mean_diff'], y[up],   c='#e74c3c',   s=25, alpha=0.8,
               label=f'Up in {label_a} ({up.sum()})')
    ax.scatter(da.loc[down, 'mean_diff'], y[down], c='#3498db',   s=25, alpha=0.8,
               label=f'Up in {label_b} ({down.sum()})')
    for idx in da[up].nlargest(n_label, 'neg_log10_padj').index:
        ax.annotate(idx, (da.loc[idx, 'mean_diff'], y[idx]), fontsize=7, ha='center', va='bottom')
    for idx in da[down].nlargest(n_label, 'neg_log10_padj').index:
        ax.annotate(idx, (da.loc[idx, 'mean_diff'], y[idx]), fontsize=7, ha='center', va='bottom')
    ax.axhline(-np.log10(fdr_thresh), ls='--', c='grey', lw=0.8)
    ax.axvline( diff_thresh, ls='--', c='grey', lw=0.5)
    ax.axvline(-diff_thresh, ls='--', c='grey', lw=0.5)
    ax.set_xlabel(f'Mean arcsinh diff ({label_a} - {label_b})')
    ax.set_ylabel(f'-log10(FDR p) [cap={y_cap}]')
    ax.set_title(title)
    ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()


def heatmap_top_diff(X_df, da, group_series, fdr_thresh=FDR_THRESH, n_top=20, title='',
                     group_labels=None):
    """group_labels: dict mapping original group values -> display names."""
    top_up   = da[da['pval_adj'] < fdr_thresh].nlargest(n_top, 'mean_diff').index.tolist()
    top_down = da[da['pval_adj'] < fdr_thresh].nsmallest(n_top, 'mean_diff').index.tolist()
    feats    = top_up + top_down[::-1]
    if not feats:
        print('No significant features to plot'); return
    # Compute per-group means explicitly — avoids all Categorical/groupby ghost-group issues
    grp = np.array(group_series.astype(str))          # plain numpy string array
    unique_groups = sorted(dict.fromkeys(grp))         # preserve order, deduplicated
    rows = {}
    for g in unique_groups:
        label = group_labels.get(g, g) if group_labels else g
        rows[label] = X_df[feats].iloc[grp == g].mean()
    hm_mean = pd.DataFrame(rows).T                    # shape: (n_groups, n_feats)
    hm_z    = (hm_mean - hm_mean.mean()) / hm_mean.std()
    g = sns.clustermap(hm_z.T, cmap='RdBu_r', center=0, figsize=(8, 12),
                       row_cluster=False, col_cluster=False,
                       yticklabels=True, xticklabels=True,
                       linewidths=0.3, linecolor='white',
                       cbar_kws={'label': 'z-scored mean arcsinh'})
    g.fig.suptitle(title, y=1.01, fontsize=12, fontweight='bold')
    plt.show()


### 4a. Abundance comparison

In [ ]:
# Classical B-cell surface markers — excluded from T-cell abundance analysis
B_CELL_MARKERS = {
    'CD19', 'CD20', 'CD22', 'CD79a', 'CD79b', 'CD24', 'CD38',
    'CD10', 'IgD', 'IgM', 'CD138', 'CD21', 'CD23', 'CD40',
    'CD185', 'CD267', 'CD268', 'CD269', 'BCMA',
}

NALM_LABEL    = 'NALM-6 co-culture'
HEALTHY_LABEL = 'Healthy B co-culture'
GROUP_LABELS  = {
    'NALM-6 + healthy T':      NALM_LABEL,
    'healthy B + healthy T':   HEALTHY_LABEL,
}

nalm_mask = (adata_6h.obs['cell_system'] == 'NALM-6 + healthy T').values

ab_df_all = pd.DataFrame(
    np.array(adata_6h.layers['arcsinh'], dtype=np.float32),
    index=adata_6h.obs_names, columns=adata_6h.var_names
)
# Drop B cell markers (not expressed on T cells; would confound the analysis)
t_cell_feats = [c for c in ab_df_all.columns if c not in B_CELL_MARKERS]
ab_df = ab_df_all[t_cell_feats]
print(f'Features after B-marker exclusion: {len(t_cell_feats)} / {ab_df_all.shape[1]}')
print(f'Removed: {sorted(B_CELL_MARKERS & set(ab_df_all.columns))}')

# ── CD4 abundance ────────────────────────────────────────────────────────
cd4_mask  = adata_6h.obs['cell_type_annot'] == 'CD4'
ab_cd4    = ab_df.loc[cd4_mask]
nalm_cd4  = nalm_mask[cd4_mask.values]
da_cd4    = diff_analysis(ab_cd4, nalm_cd4)
group_cd4 = adata_6h.obs.loc[cd4_mask, 'cell_system']

print(f'CD4 Abundance — significant (FDR<{FDR_THRESH}): {(da_cd4["pval_adj"]<FDR_THRESH).sum()} / {len(da_cd4)}')

volcano_plot(
    da_cd4, diff_thresh=0.3,
    label_a=NALM_LABEL, label_b=HEALTHY_LABEL,
    title=f'6h Mock — CD4 T cell abundance ({NALM_LABEL} vs {HEALTHY_LABEL})'
)

heatmap_top_diff(ab_cd4, da_cd4, group_cd4, group_labels=GROUP_LABELS,
                 title='6h Mock CD4 Abundance — NALM-6 vs Healthy B', n_top=10)

print('Top 10 CD4 abundance markers (NALM-6 > Healthy B):')
print(da_cd4[da_cd4['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())
print('\nTop 10 CD4 abundance markers (Healthy B > NALM-6):')
print(da_cd4[da_cd4['pval_adj'] < FDR_THRESH].nsmallest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())

# ── CD8 abundance ────────────────────────────────────────────────────────
cd8_mask  = adata_6h.obs['cell_type_annot'] == 'CD8'
ab_cd8    = ab_df.loc[cd8_mask]
nalm_cd8  = nalm_mask[cd8_mask.values]
da_cd8    = diff_analysis(ab_cd8, nalm_cd8)
group_cd8 = adata_6h.obs.loc[cd8_mask, 'cell_system']

print(f'\nCD8 Abundance — significant (FDR<{FDR_THRESH}): {(da_cd8["pval_adj"]<FDR_THRESH).sum()} / {len(da_cd8)}')

volcano_plot(
    da_cd8, diff_thresh=0.3,
    label_a=NALM_LABEL, label_b=HEALTHY_LABEL,
    title=f'6h Mock — CD8 T cell abundance ({NALM_LABEL} vs {HEALTHY_LABEL})'
)

heatmap_top_diff(ab_cd8, da_cd8, group_cd8, group_labels=GROUP_LABELS,
                 title='6h Mock CD8 Abundance — NALM-6 vs Healthy B', n_top=10)

print('Top 10 CD8 abundance markers (NALM-6 > Healthy B):')
print(da_cd8[da_cd8['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())
print('\nTop 10 CD8 abundance markers (Healthy B > NALM-6):')
print(da_cd8[da_cd8['pval_adj'] < FDR_THRESH].nsmallest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())


### 4b. Spatial colocalization comparison

In [ ]:
sp_df = adata_6h.obsm[spatial_rep].copy()
if not isinstance(sp_df, pd.DataFrame):
    sp_df = pd.DataFrame(sp_df, index=adata_6h.obs_names)
else:
    sp_df.index = adata_6h.obs_names

# ── CD4 spatial ──────────────────────────────────────────────────────────
sp_cd4    = sp_df.loc[cd4_mask]
da_sp_cd4 = diff_analysis(sp_cd4, nalm_cd4)

print(f'CD4 Spatial - significant (FDR<{FDR_THRESH}): {(da_sp_cd4["pval_adj"]<FDR_THRESH).sum()} / {len(da_sp_cd4)}')

volcano_plot(
    da_sp_cd4, diff_thresh=0.1,
    label_a='NALM-6 + healthy T', label_b='Healthy B + healthy T',
    title='6h Mock — CD4 T cell spatial (NALM-6 vs Healthy B co-culture)', n_label=8
)

heatmap_top_diff(sp_cd4, da_sp_cd4, group_cd4,
                 title='6h Mock CD4 Spatial — NALM-6 vs Healthy B', n_top=10)

print('Top 10 CD4 spatial pairs (NALM-6 > Healthy B):')
print(da_sp_cd4[da_sp_cd4['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())
print('\nTop 10 CD4 spatial pairs (Healthy B > NALM-6):')
print(da_sp_cd4[da_sp_cd4['pval_adj'] < FDR_THRESH].nsmallest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())

# ── CD8 spatial ──────────────────────────────────────────────────────────
sp_cd8    = sp_df.loc[cd8_mask]
da_sp_cd8 = diff_analysis(sp_cd8, nalm_cd8)

print(f'\nCD8 Spatial - significant (FDR<{FDR_THRESH}): {(da_sp_cd8["pval_adj"]<FDR_THRESH).sum()} / {len(da_sp_cd8)}')

volcano_plot(
    da_sp_cd8, diff_thresh=0.1,
    label_a='NALM-6 + healthy T', label_b='Healthy B + healthy T',
    title='6h Mock — CD8 T cell spatial (NALM-6 vs Healthy B co-culture)', n_label=8
)

heatmap_top_diff(sp_cd8, da_sp_cd8, group_cd8,
                 title='6h Mock CD8 Spatial — NALM-6 vs Healthy B', n_top=10)

print('Top 10 CD8 spatial pairs (NALM-6 > Healthy B):')
print(da_sp_cd8[da_sp_cd8['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())
print('\nTop 10 CD8 spatial pairs (Healthy B > NALM-6):')
print(da_sp_cd8[da_sp_cd8['pval_adj'] < FDR_THRESH].nsmallest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())

### 4c. Top hits summary

In [ ]:
print('=== Top abundance markers — T cells in NALM-6 > Healthy B co-culture (6h Mock) ===')
print('\nCD4:')
print(da_cd4[da_cd4['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())
print('\nCD8:')
print(da_cd8[da_cd8['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())

print('\n=== Top spatial pairs — T cells in NALM-6 > Healthy B co-culture (6h Mock) ===')
print('\nCD4:')
print(da_sp_cd4[da_sp_cd4['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())
print('\nCD8:')
print(da_sp_cd8[da_sp_cd8['pval_adj'] < FDR_THRESH].nlargest(10, 'mean_diff')[['mean_diff', 'pval_adj']].to_string())